# VisionBridge — automatic real-video validation

This notebook downloads ISL-CSLTR, selects real sentence-level videos automatically, extracts pose + face + both hand skeletons, loads the current checkpoint, and reports ground truth, prediction, confidence, CER, blank/space ratios, and tensor contracts.

It never fabricates a prediction and never treats an empty/space-only output as success.

In [ ]:
import os,sys,subprocess,shutil,glob,hashlib,re,random,csv,textwrap
from pathlib import Path
import torch
REPO=Path('/content/VisionBridge')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
BACKEND=REPO/'backend'; sys.path.insert(0,str(BACKEND)); os.chdir(REPO)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('HEAD:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Device:',DEVICE)


In [ ]:
# Install isolated extraction runtime.
subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub'],check=True)
uv=shutil.which('uv') or '/usr/local/bin/uv'
MP_ENV=Path('/content/visionbridge_mp312'); MP_PY=MP_ENV/'bin'/'python'; MPLCONFIG=Path('/content/visionbridge_mplconfig'); MPLCONFIG.mkdir(parents=True,exist_ok=True)
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['MPLCONFIGDIR']=str(MPLCONFIG)
if not MP_ENV.exists(): subprocess.run([uv,'python','install','3.12'],check=True); subprocess.run([uv,'venv','--python','3.12',str(MP_ENV)],check=True)
subprocess.run([uv,'pip','install','--python',str(MP_PY),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','matplotlib'],check=True,env=env)
probe=subprocess.run([str(MP_PY),'-c','from mediapipe.python.solutions import holistic; import mediapipe; print(mediapipe.__version__)'],capture_output=True,text=True,env=env)
if probe.returncode!=0 or probe.stdout.strip().splitlines()[-1]!='0.10.21': raise RuntimeError(probe.stderr or probe.stdout)
print('MediaPipe:',probe.stdout.strip().splitlines()[-1])


In [ ]:
# Download dataset and automatically choose several real videos.
import kagglehub
root=Path(kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset'))
video_roots=[p for p in root.rglob('Videos_Sentence_Level') if p.is_dir()]
if len(video_roots)!=1: raise RuntimeError(f'Expected one sentence-level video root; found {video_roots}')
VIDEO_ROOT=video_roots[0]
videos=sorted(set(p for p in VIDEO_ROOT.rglob('*') if p.is_file() and p.suffix.casefold() in {'.mp4','.avi','.mov','.mkv'}))
if len(videos)<3: raise RuntimeError('Need at least three sentence-level videos for validation.')
rng=random.Random(2026); selected=rng.sample(videos,min(5,len(videos)))
print('Dataset:',root)
for p in selected: print('SELECTED:',p.parent.name,'/',p.name)


In [ ]:
# Extract hand-aware features for each selected video using the repository extractor.
HELPER=Path('/content/vb_validate_one.py')
HELPER.write_text(textwrap.dedent('''
from pathlib import Path
import sys, numpy as np
repo=Path(sys.argv[1]); video=Path(sys.argv[2]); out=Path(sys.argv[3]); sys.path.insert(0,str(repo/'backend'))
from mediapipe.python.solutions import holistic
from scripts.extract_keypoints import extract_clip_keypoints_with_hands
uid='sample'
with holistic.Holistic(static_image_mode=False, model_complexity=1, refine_face_landmarks=False) as h:
    pose,face,left,right=extract_clip_keypoints_with_hands(str(video),h)
assert pose.shape[1]==132 and face.shape[1]==1404 and left.shape[1]==63 and right.shape[1]==63
assert len({pose.shape[0],face.shape[0],left.shape[0],right.shape[0]})==1 and pose.shape[0]>0
np.save(out/'pose.npy',pose); np.save(out/'face.npy',face); np.save(out/'left.npy',left); np.save(out/'right.npy',right)
'''),encoding='utf-8')
VALIDATION_ROOT=Path('/content/visionbridge_validation'); shutil.rmtree(VALIDATION_ROOT,ignore_errors=True); VALIDATION_ROOT.mkdir(parents=True)
results=[]
for i,video in enumerate(selected):
    out=VALIDATION_ROOT/str(i); out.mkdir()
    r=subprocess.run([str(MP_PY),str(HELPER),str(REPO),str(video),str(out)],capture_output=True,text=True,env=env)
    if r.returncode!=0: raise RuntimeError(f'Extraction failed for {video}:\n{r.stderr}')
    results.append((video,out))
    print(f'EXTRACTED {i+1}/{len(selected)}:',video.name)


In [ ]:
# Load current checkpoint and evaluate every selected real video.
from app.models.base_model import load_frozen_base_model
from app.services import inference_service
WEIGHTS=REPO/'backend/app/models/weights/base_model.pt'
if not WEIGHTS.exists(): raise FileNotFoundError(f'Missing checkpoint: {WEIGHTS}')
state=torch.load(WEIGHTS,map_location='cpu',weights_only=True)
if 'left_hand_encoder.input_proj.0.weight' not in state: raise RuntimeError('Current checkpoint is legacy pose+face. Retrain the new hand-aware model before real-video validation.')
model=load_frozen_base_model(str(WEIGHTS)).to(DEVICE).eval()
inference_service._id_to_token=inference_service._load_vocab(str(WEIGHTS))
def cer(pred,target):
    a=pred.lower(); b=target.lower(); prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]/max(len(b),1)
for video,out in results:
    import numpy as np
    pose=torch.from_numpy(np.load(out/'pose.npy')).float(); face=torch.from_numpy(np.load(out/'face.npy')).float(); left=torch.from_numpy(np.load(out/'left.npy')).float(); right=torch.from_numpy(np.load(out/'right.npy')).float()
    frames=pose.shape[0]; pose=pose.unsqueeze(0).to(DEVICE); face=face.unsqueeze(0).to(DEVICE); left=left.unsqueeze(0).to(DEVICE); right=right.unsqueeze(0).to(DEVICE); length=torch.tensor([frames],device=DEVICE)
    with torch.inference_mode(): logits=model(pose,face,left,right,length)
    pred,conf=inference_service.decode_logits(logits); truth=video.parent.name.replace('_',' ').strip(); clean='' if pred=='(no sign detected)' else pred.strip(); ids=logits[0,:frames].argmax(-1); space_id=inference_service._id_to_token and next((i for i,t in inference_service._id_to_token.items() if t==' '),37); blank_ratio=float((ids==0).float().mean()); space_ratio=float((ids==space_id).float().mean())
    print('\n'+'='*72); print('VIDEO:',video.name); print('GROUND TRUTH:',truth); print('PREDICTED:',pred); print('CONFIDENCE:',round(conf,4)); print('BLANK RATIO:',round(blank_ratio,4)); print('SPACE RATIO:',round(space_ratio,4)); print('CER:',round(cer(clean,truth),4)); print('FRAMES:',frames); print('POSE:',tuple(pose.shape)); print('FACE:',tuple(face.shape)); print('LEFT HAND:',tuple(left.shape)); print('RIGHT HAND:',tuple(right.shape))


In [ ]:
print('\n'+'='*72)
print('REAL-VIDEO VALIDATION COMPLETE')
print('A result is informational only; the model is accepted only if multiple real videos show non-trivial, semantically useful output.')
